In [ ]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import torch
import random

class LoadSeqDataset(Dataset):
    def __init__(self, file_path: str, label: int, selected_features: list, seq_num=28, gap=5):
        """
        An optimized dataset loader that uses vectorized operations for speed.
        """
        self.seq_num = seq_num
        self.gap = gap
        
        # Load the necessary data directly into NumPy arrays for performance
        df = pd.read_csv(file_path, usecols=selected_features + ['label'], engine='python')
        
        # Process features and labels
        features_data = df[selected_features].values.astype(np.float32)
        # Create a single label for the entire file's content
        is_contact_file = 1 if label > 0 else 0
        labels_data = (df['label'].values > 0).astype(np.int8) * is_contact_file
        
        self.sequences, self.labels = self._make_sequences_vectorized(features_data, labels_data)

    def _make_sequences_vectorized(self, features, labels):
        """
        Creates sequences using efficient NumPy array manipulation.
        """
        # Create overlapping windows of the specified sequence length
        # Shape will be (num_sequences, num_features, seq_num)
        shape = (features.shape[0] - self.seq_num + 1, features.shape[1], self.seq_num)
        strides = (features.strides[0], features.strides[1], features.strides[0])
        windows = np.lib.stride_tricks.as_strided(features, shape=shape, strides=strides)
        
        # Get the label for the last timestep in each window
        sequence_labels = labels[self.seq_num - 1:]
        
        # Apply the gap by subsampling the windows and labels
        return windows[::self.gap], sequence_labels[::self.gap]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        # The data is already in the correct [dof, seq_num] format
        features = torch.from_numpy(self.sequences[idx])
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        return features, target

# The LoadDatasets class remains the same
class LoadDatasets(Dataset):
    def __init__(self, data_path:str, dict_label = None):
        if dict_label is None:
            dict_label = {'a': 7, 'b': 6, 'c': 5, 'd': 4, 'e': 3, 'f': 2, 'g': 1}
        
        self.samples = []
        self.class_to_idx = {}

        for class_name in sorted(os.listdir(data_path)):
            class_dir = os.path.join(data_path, class_name)
            if os.path.isdir(class_dir) and class_name in dict_label:
                label = dict_label[class_name]
                self.class_to_idx[class_name] = label
                for file_name in os.listdir(class_dir):
                    file_path = os.path.join(class_dir, file_name)
                    if os.path.isfile(file_path):
                        self.samples.append((file_path, label))
                        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq_path, label = self.samples[idx]
        return seq_path, label

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, ConcatDataset
import os
import logging
import glob
from multiprocessing import Pool, cpu_count
import time

from models.cnnLSTM_contactDetection import cnnLSTM

# Helper function for parallel data loading
def load_dataset_worker(args):
    file_path, label, selected_features, seq_num, gap = args
    return LoadSeqDataset(file_path, label, selected_features, seq_num, gap)

def train_model(full_dataset,model,  n_epochs=50, batch_size=64, learning_rate=0.001, model_path='cnn_lstm_model'):
    # This function remains the same as your provided script
    # ... (training and validation loop) ...
    train_size = int(0.5 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, 
                              num_workers=min(4, os.cpu_count()), pin_memory=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=min(4, os.cpu_count()), pin_memory=True)
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3, verbose=True)
    
    scaler_amp = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best_val_accuracy = 0.0

    logging.info("--- Starting Optimized Model Training ---")
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True).float()
            
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            optimizer.zero_grad(set_to_none=True)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
            
            running_loss += loss.item()
            
        avg_train_loss = running_loss / len(train_loader)
        
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True).float()
                
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    outputs = model(inputs)
                
                val_loss += criterion(outputs, labels).item()
                predicted = (torch.sigmoid(outputs) > 0.5).int()
                total += labels.size(0)
                correct += (predicted == labels.int()).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        
        logging.info(f'Epoch [{epoch+1}/{n_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        
        scheduler.step(avg_val_loss)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), f'{model_path}_accuracy{best_val_accuracy}.pth')
            logging.info(f"New best model saved with accuracy: {best_val_accuracy:.2f}%")
            
    return model, best_val_accuracy

if __name__ == '__main__':
    # --- Configuration ---
    project_root = os.getcwd().replace('pipelines','')

    data_name = 'franka_main'
    dof = 7
    hidden_size = 128
    seq_num = 100
    num_layers = 1

    gap = 5
    batch_size = 71
    n_epochs = 10

    log_dir = f'{project_root}/pipelines/trained_models/{data_name}/contact_detection_v2/{batch_size}/'
    os.makedirs(log_dir, exist_ok=True)
    model_name = f'numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}'

    log_file = os.path.join(log_dir, f'training_log_{time.time()}.txt')
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', handlers=[logging.StreamHandler(), logging.FileHandler(log_file)])
    
    logging.info(f"Logging to {log_file}")

    # --- Setup Logging and Reproducibility ---
    # ... (logging and seeding is unchanged) ...

    # --- Optimized Data Loading from Labeled CSVs ---
    data_directory = os.path.join(project_root, 'dataset', data_name, 'labeled_data')
    dict_label = {'link7': 7, 'link6': 6, 'link5': 5, 'link4': 4, 'link3': 3, 'link2': 2, 'link1': 1, 'no_contact': 0}
    selected_features = [f'e{i}' for i in range(dof)]
    
    all_csv_files = glob.glob(os.path.join(data_directory, '**', '*.csv'), recursive=True)
    
    if not all_csv_files:
        logging.error(f"No CSV files found in '{data_directory}'. Please check the path.")
    else:
        # Create a list of arguments for the parallel worker function
        tasks = []
        for file in all_csv_files:
            label = 1 if 'no_contact' not in file else 0
            tasks.append((file, torch.tensor(label), selected_features, seq_num, gap))

        # Use a multiprocessing Pool to load datasets in parallel
        logging.info(f"Starting parallel data loading with {cpu_count()} workers...")
        with Pool(processes=cpu_count()) as pool:
            all_datasets = pool.map(load_dataset_worker, tasks)
        
        # Combine them into a single dataset
        master_dataset = ConcatDataset(all_datasets)
        logging.info(f"Successfully loaded and combined data from {len(all_csv_files)} files into a dataset with {len(master_dataset)} samples.")
        
        # --- Train the Model ---
        model = cnnLSTM(num_features_joints=seq_num, hidden_size=hidden_size, num_layers=num_layers)

        trained_model, accuracy  = train_model(full_dataset=master_dataset,model=model, n_epochs=n_epochs, model_path = f'{log_dir}{model_name}')
        

2025-09-19 18:57:54,320 - INFO - Logging to /home/rzma/myProjects/contactInterpretation//pipelines/trained_models/franka_main/contact_detection_v2/71/training_log_1758301074.320228.txt
2025-09-19 18:57:54,323 - INFO - Starting parallel data loading with 48 workers...


2025-09-19 18:57:56,940 - INFO - Successfully loaded and combined data from 501 files into a dataset with 327053 samples.
2025-09-19 18:58:00,349 - INFO - --- Starting Optimized Model Training ---
2025-09-19 18:58:22,323 - INFO - Epoch [1/10], Train Loss: 0.3332, Val Loss: 0.2705, Val Acc: 89.51%
2025-09-19 18:58:22,334 - INFO - New best model saved with accuracy: 89.51%
2025-09-19 18:58:44,330 - INFO - Epoch [2/10], Train Loss: 0.2571, Val Loss: 0.2368, Val Acc: 90.75%
2025-09-19 18:58:44,340 - INFO - New best model saved with accuracy: 90.75%
2025-09-19 18:59:06,209 - INFO - Epoch [3/10], Train Loss: 0.2248, Val Loss: 0.2514, Val Acc: 90.16%
2025-09-19 18:59:28,837 - INFO - Epoch [4/10], Train Loss: 0.2036, Val Loss: 0.2056, Val Acc: 92.13%
2025-09-19 18:59:28,847 - INFO - New best model saved with accuracy: 92.13%
2025-09-19 18:59:53,049 - INFO - Epoch [5/10], Train Loss: 0.1904, Val Loss: 0.2165, Val Acc: 91.46%
2025-09-19 19:00:17,057 - INFO - Epoch [6/10], Train Loss: 0.1798, Val

In [4]:
trained_model, accuracy  = train_model(full_dataset=master_dataset,model=model, n_epochs=n_epochs, model_path = f'{log_dir}{model_name}')

2025-09-19 19:08:25,836 - INFO - --- Starting Optimized Model Training ---
2025-09-19 19:08:47,583 - INFO - Epoch [1/10], Train Loss: 0.1265, Val Loss: 0.1599, Val Acc: 94.35%
2025-09-19 19:08:47,594 - INFO - New best model saved with accuracy: 94.35%
2025-09-19 19:09:09,886 - INFO - Epoch [2/10], Train Loss: 0.1199, Val Loss: 0.1390, Val Acc: 94.72%
2025-09-19 19:09:09,896 - INFO - New best model saved with accuracy: 94.72%
2025-09-19 19:09:31,616 - INFO - Epoch [3/10], Train Loss: 0.1147, Val Loss: 0.1541, Val Acc: 94.22%
2025-09-19 19:09:53,971 - INFO - Epoch [4/10], Train Loss: 0.1090, Val Loss: 0.1532, Val Acc: 94.77%
2025-09-19 19:09:53,981 - INFO - New best model saved with accuracy: 94.77%
2025-09-19 19:10:17,317 - INFO - Epoch [5/10], Train Loss: 0.1045, Val Loss: 0.1467, Val Acc: 94.79%
2025-09-19 19:10:17,327 - INFO - New best model saved with accuracy: 94.79%
2025-09-19 19:10:40,295 - INFO - Epoch [6/10], Train Loss: 0.1015, Val Loss: 0.2227, Val Acc: 92.27%


Epoch     6: reducing learning rate of group 0 to 5.0000e-04.


2025-09-19 19:11:03,169 - INFO - Epoch [7/10], Train Loss: 0.0815, Val Loss: 0.1356, Val Acc: 95.27%
2025-09-19 19:11:03,179 - INFO - New best model saved with accuracy: 95.27%
2025-09-19 19:11:25,338 - INFO - Epoch [8/10], Train Loss: 0.0758, Val Loss: 0.1341, Val Acc: 95.27%
2025-09-19 19:11:49,910 - INFO - Epoch [9/10], Train Loss: 0.0719, Val Loss: 0.1478, Val Acc: 94.83%
2025-09-19 19:12:12,468 - INFO - Epoch [10/10], Train Loss: 0.0690, Val Loss: 0.1343, Val Acc: 95.50%
2025-09-19 19:12:12,478 - INFO - New best model saved with accuracy: 95.50%
